# ExoScout v0.2

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_data import load_data

df = load_data()

In [2]:
filtered_df = df[df["tfopwg_disp"].isin(["CP", "KP", "FP"])].copy()

mapping = {
    "CP": 1,
    "KP": 1,
    "FP": 0
}

filtered_df["target"] = filtered_df["tfopwg_disp"].map(mapping)

In [3]:
features = [
    "pl_orbper",
    "pl_trandurh",
    "pl_trandep",
    "st_tmag",
    "st_teff",
    "st_logg",
    "st_rad"
]

## 1. Missingness Analysis by Class

In [4]:
missing_by_class = (
    filtered_df
    .groupby("target")[features]
    .apply(lambda group: group.isna().mean())
)

missing_by_class

,pl_orbper,pl_trandurh,pl_trandep,st_tmag,st_teff,st_logg,st_rad
target,,,,,,,
0,0.008765,0.0,0.0,0.0,0.029482,0.162550,0.096414
1,0.012677,0.0,0.0,0.0,0.000000,0.009694,0.007457


### Finding

Missing values are not evenly distributed across the two classes.

In particular, `st_logg` is missing in about 16.3% of false positives but only about 1.0% of planets, while `st_rad` is missing in about 9.6% of false positives and 0.7% of planets.

This suggests that dropping every row containing missing values may disproportionately remove false positives and potentially bias the training dataset.

In [5]:
complete_by_class = (
    filtered_df
    .assign(is_complete=filtered_df[features].notna().all(axis=1))
    .groupby("target")["is_complete"]
    .mean()
)

complete_by_class

target
0    0.804781
1    0.972409
Name: is_complete, dtype: float64

### Complete-case analysis

Using complete cases only retains about 80.5% of false positives but 97.2% of planets.

Therefore, the v0.1 `dropna()` strategy disproportionately removes false positives. This can alter the class distribution and potentially bias both training and evaluation.

For v0.2, missing values will therefore be handled through imputation rather than row deletion.

In [6]:
X = filtered_df[features]
Y = filtered_df["target"]

For v0.2, the model uses the full filtered dataset, including rows with missing feature values. Missing values will be handled with imputation instead of dropping rows.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [8]:
X_train.shape, X_test.shape

((2076, 7), (520, 7))

In [9]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

In [10]:
import numpy as np

np.isnan(X_train_imputed).sum(), np.isnan(X_test_imputed).sum()

(np.int64(0), np.int64(0))

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion matrix:")
print(cm)
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.676923076923077
Confusion matrix:
[[160  91]
 [ 77 192]]

              precision    recall  f1-score   support

           0       0.68      0.64      0.66       251
           1       0.68      0.71      0.70       269

    accuracy                           0.68       520
   macro avg       0.68      0.68      0.68       520
weighted avg       0.68      0.68      0.68       520



In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])


In [15]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [17]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [18]:
results = cross_validate(
    pipeline,
    X,
    Y,
    cv=cv,
    scoring=scoring
)

In [19]:
results

{'fit_time': array([0.03849912, 0.01666999, 0.08957219, 0.03481698, 0.02364111]),
 'score_time': array([0.02774191, 0.03262997, 0.09767795, 0.075634  , 0.02300906]),
 'test_accuracy': array([0.68076923, 0.68208092, 0.6743738 , 0.67630058, 0.71098266]),
 'test_precision': array([0.66885246, 0.67697595, 0.68679245, 0.68656716, 0.70774648]),
 'test_recall': array([0.75836431, 0.73507463, 0.67910448, 0.68656716, 0.75      ]),
 'test_f1': array([0.71080139, 0.70483005, 0.68292683, 0.68656716, 0.72826087])}

In [20]:
accuracy_mean = results["test_accuracy"].mean()
accuracy_std = results["test_accuracy"].std()

accuracy_mean, accuracy_std

(np.float64(0.6849014376760042), np.float64(0.01334216295051377))

In [21]:
precision_mean = results["test_precision"].mean()
precision_std = results["test_precision"].std()

recall_mean = results["test_recall"].mean()
recall_std = results["test_recall"].std()

f1_mean = results["test_f1"].mean()
f1_std = results["test_f1"].std()

precision_mean, precision_std, recall_mean, recall_std, f1_mean, f1_std

(np.float64(0.6853868999832217),
 np.float64(0.013018842283727822),
 np.float64(0.7218221161848749),
 np.float64(0.032780062710046604),
 np.float64(0.7026772620816201),
 np.float64(0.016581748093851403))

### Logistic Regression — 5-Fold Cross-Validation

Using median imputation, standard scaling and Logistic Regression:

- Accuracy: 68.5% ± 1.3%
- Precision (planet): 68.5% ± 1.3%
- Recall (planet): 72.2% ± 3.3%
- F1-score (planet): 70.3% ± 1.7%

These results suggest that the single-split v0.1 accuracy of ~71.7% was likely somewhat optimistic. Cross-validation provides a more stable estimate of baseline performance.

### Experiment: Missingness Indicators

The previous analysis showed that missing values are not evenly distributed between the two classes.

For example, `st_logg` and `st_rad` are missing much more frequently among false positives than among confirmed/known planets.

In the previous pipeline, missing values were replaced with the median. However, after imputation, the model could no longer distinguish between an original value and a value that had been inserted by the imputer.

To preserve this information, this experiment adds a binary missingness indicator for features containing missing values.

For each affected feature, the indicator records whether the original value was missing:

- `0` → the original value was available
- `1` → the original value was missing and was imputed

The rest of the pipeline remains unchanged:

1. Median imputation
2. Missingness indicators
3. Standard scaling
4. Logistic Regression
5. 5-fold stratified cross-validation

By keeping the model and cross-validation setup unchanged, we can isolate the effect of adding missingness information.

In [22]:
pipeline_with_indicators = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

In [23]:
results_with_indicators = cross_validate(
    pipeline_with_indicators,
    X,
    Y,
    cv=cv,
    scoring=scoring
)

In [24]:
results_with_indicators["test_accuracy"]

array([0.73076923, 0.71868979, 0.73603083, 0.72447013, 0.76685934])

In [25]:
indicator_accuracy_mean = results_with_indicators["test_accuracy"].mean()
indicator_accuracy_std = results_with_indicators["test_accuracy"].std()

indicator_precision_mean = results_with_indicators["test_precision"].mean()
indicator_precision_std = results_with_indicators["test_precision"].std()

indicator_recall_mean = results_with_indicators["test_recall"].mean()
indicator_recall_std = results_with_indicators["test_recall"].std()

indicator_f1_mean = results_with_indicators["test_f1"].mean()
indicator_f1_std = results_with_indicators["test_f1"].std()

(
    indicator_accuracy_mean,
    indicator_accuracy_std,
    indicator_precision_mean,
    indicator_precision_std,
    indicator_recall_mean,
    indicator_recall_std,
    indicator_f1_mean,
    indicator_f1_std
)

(np.float64(0.7353638654216688),
 np.float64(0.01679415943622169),
 np.float64(0.7243144020696367),
 np.float64(0.0184633268775179),
 np.float64(0.7889502302613327),
 np.float64(0.03399449644640934),
 np.float64(0.754692573085974),
 np.float64(0.01720650319950119))

### Result

Adding missingness indicators produced a consistent improvement across all 5 folds.

Compared with median imputation alone:

- Accuracy increased from 68.5% ± 1.3% to 73.5% ± 1.7%
- Precision increased from 68.5% ± 1.3% to 72.4% ± 1.8%
- Recall increased from 72.2% ± 3.3% to 78.9% ± 3.4%
- F1-score increased from 70.3% ± 1.7% to 75.5% ± 1.7%

This suggests that the pattern of missing values contains useful predictive information and should be preserved rather than discarded.

### Experiment Log

The goal of this table is to compare preprocessing choices under the same 5-fold stratified cross-validation setup.

In [26]:
import pandas as pd

experiment_log = pd.DataFrame({
    "Experiment": [
        "Median imputation",
        "Median imputation + missing indicators"
    ],
    "Accuracy": [
        accuracy_mean,
        indicator_accuracy_mean
    ],
    "Precision": [
        precision_mean,
        indicator_precision_mean
    ],
    "Recall": [
        recall_mean,
        indicator_recall_mean
    ],
    "F1": [
        f1_mean,
        indicator_f1_mean
    ]
})

experiment_log

,Experiment,Accuracy,Precision,Recall,F1
0,Median imputation,0.684901,0.685387,0.721822,0.702677
1,Median imputation + missing indicators,0.735364,0.724314,0.788950,0.754693


In [27]:
experiment_log.round(3)

,Experiment,Accuracy,Precision,Recall,F1
0,Median imputation,0.685,0.685,0.722,0.703
1,Median imputation + missing indicators,0.735,0.724,0.789,0.755


### Fold-by-Fold Comparison

To verify whether missingness indicators improve performance consistently, the accuracy of the two preprocessing strategies is compared on the same five cross-validation folds.

In [28]:
fold_comparison = pd.DataFrame({
    "Fold": range(1, 6),
    "Median Imputation": results["test_accuracy"],
    "With Missing Indicators": results_with_indicators["test_accuracy"]
})

fold_comparison["Difference"] = (
    fold_comparison["With Missing Indicators"]
    - fold_comparison["Median Imputation"]
)

fold_comparison.round(3)

,Fold,Median Imputation,With Missing Indicators,Difference
0,1,0.681,0.731,0.050
1,2,0.682,0.719,0.037
2,3,0.674,0.736,0.062
3,4,0.676,0.724,0.048
4,5,0.711,0.767,0.056


### Fold-by-Fold Result

Missingness indicators improved accuracy in all five cross-validation folds.

The improvement ranged from 3.7 to 6.2 percentage points, suggesting that the gain is consistent across different train/validation partitions rather than being driven by a single favorable split.

### Methodological Note

The initial missingness analysis was performed on the full labeled dataset before cross-validation.

Therefore, although preprocessing itself was correctly fitted only within each training fold, the decision to test missingness indicators was informed by patterns observed across the full dataset.

This may introduce some model-selection bias and should be considered a limitation of the v0.2 evaluation.

Future experiments will use a separate final holdout set that remains untouched during model development.

## v0.2 Conclusion

The v0.2 experiments showed that missingness patterns are strongly associated with the target class.

Using median imputation alone produced an average accuracy of 68.5% under 5-fold stratified cross-validation.

Adding binary missingness indicators improved average accuracy to 73.5%. The improvement was observed in all five folds, ranging from 3.7 to 6.2 percentage points, and was also reflected in precision, recall, and F1-score.

These results suggest that missing-data patterns contain useful predictive information in the TOI dataset and should be preserved rather than discarded during preprocessing.

However, the initial missingness analysis was performed on the full labeled dataset before cross-validation. Therefore, the decision to test missingness indicators was partially informed by data later used in validation folds. While preprocessing itself remained leakage-free, this introduces a potential model-selection bias and limits how independently the v0.2 performance estimate should be interpreted.

Future model development will therefore use a separate final holdout set that remains untouched during experimentation.